## Connect Google Drive

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ModuleNotFoundError:
    print("Running outside Google Colab. Skipping Google Drive mounting.")

Running outside Google Colab. Skipping Google Drive mounting.


In [2]:
import os
HOME = r"C:\Users\HP\Desktop\RockPaperScissor"
os.makedirs(HOME, exist_ok=True)
os.chdir(HOME)
print(HOME)

C:\Users\HP\Desktop\RockPaperScissor


## Install YOLO11 via Ultralytics

In [3]:
%pip install ultralytics supervision roboflow

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

Ultralytics 8.4.113  Python-3.11.9 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
Setup complete  (8 CPUs, 15.8 GB RAM, 116.1/117.2 GB disk)


In [4]:
from ultralytics import YOLO
from IPython.display import display, Image
from roboflow import Roboflow

## Download Dataset

We will now connect with our roboflow account and download the required dataset and save it our drive folder

* `location`: the path where the dataset should be saved
* `overwrite`: downloads again if `True`. Reads from drive folder if `False`

In [5]:
DATASET_FOLDER = f'{HOME}/datasets'

from roboflow import Roboflow
from getpass import getpass

API_KEY = getpass("Enter your Roboflow API key: ")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("default-rpbis").project("rock-paper-scissors-sxsw-fjg7q")
version = project.version(1)

dataset = version.download(
    "yolov11",
    overwrite=False,
    location=DATASET_FOLDER
)

loading Roboflow workspace...
loading Roboflow project...


## Training

In [6]:
from ultralytics import YOLO
import os

os.chdir(HOME)

model = YOLO("yolo11s.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=10,
    imgsz=416,
    batch=8,
    workers=0,
    plots=True
)

New https://pypi.org/project/ultralytics/8.4.114 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.113  Python-3.11.9 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\HP\Desktop\RockPaperScissor\datasets/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=3

## Validate fine-tuned model

In [9]:
from ultralytics import YOLO

model = YOLO(r"C:\Users\HP\runs\detect\train-8\weights\best.pt")

metrics = model.val(data=f"{dataset.location}/data.yaml")

Ultralytics 8.4.113  Python-3.11.9 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 165.276.7 MB/s, size: 57.8 KB)
val: Scanning C:\Users\HP\Desktop\RockPaperScissor\datasets\valid\labels.cache... 604 images, 251 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 604/604  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 2.7s/it 1:422.3sss
                   all        604        418      0.833      0.852      0.882      0.652
                 Paper        139        146       0.81      0.829      0.874      0.635
                  Rock        128        150      0.901       0.85      0.876      0.648
              Scissors        118        122      0.787      0.879      0.896      0.674
Speed: 0.9ms preprocess, 148.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to C:\Users

## Inference with custom model

In [10]:
from ultralytics import YOLO

model = YOLO(r"C:\Users\HP\runs\detect\train-8\weights\best.pt")

results = model.predict(
    source=f"{dataset.location}/test/images",
    conf=0.25,
    save=True
)


image 1/329 C:\Users\HP\Desktop\RockPaperScissor\datasets\test\images\10e0gvm_jpg.rf.8dc51602cf298b899ccff72671c39fa5.jpg: 416x352 (no detections), 250.2ms
image 2/329 C:\Users\HP\Desktop\RockPaperScissor\datasets\test\images\15208484cellblock_jpg.rf.7f79be3d6c5fff044c6bdee309e8bf58.jpg: 416x416 (no detections), 410.1ms
image 3/329 C:\Users\HP\Desktop\RockPaperScissor\datasets\test\images\19171_298_298_1_0_jpg.rf.1b609cc1efc2d1218ed350786fc4047b.jpg: 416x416 (no detections), 229.2ms
image 4/329 C:\Users\HP\Desktop\RockPaperScissor\datasets\test\images\20061004021831_jpg.rf.0cf579aad1dc9bbb4df758873a57ff46.jpg: 320x416 (no detections), 246.8ms
image 5/329 C:\Users\HP\Desktop\RockPaperScissor\datasets\test\images\20220216_221550_jpg.rf.1b33846a5a9e1474e11799cbbfd264c5.jpg: 256x416 (no detections), 226.5ms
image 6/329 C:\Users\HP\Desktop\RockPaperScissor\datasets\test\images\20220216_221819_jpg.rf.5e753463d19378f56ed5851632a7f433.jpg: 256x416 1 Scissors, 165.5ms
image 7/329 C:\Users\HP\D